In [21]:
!pip install requests pandas

In [22]:
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [23]:
SEI_BASE_URL = "https://api.sei.pi.gov.br/v1"
SEI_USERNAME = "gabriel.coelho@sead.pi.gov.br"
SEI_PASSWORD = "fenek3161@SEAD"
SEI_ORGAO = "SEAD-PI"

In [24]:
# Login
login_resp = requests.post(
    f"{SEI_BASE_URL}/orgaos/usuarios/login",
    json={"Usuario": SEI_USERNAME, "Senha": SEI_PASSWORD, "Orgao": SEI_ORGAO},
    headers={"accept": "application/json", "Content-Type": "application/json"},
)
login_data = login_resp.json()
token = login_data["Token"]
print(f"Login OK - Token: {token}...")
print(f"Unidades do usuario: {login_data.get('Unidades', [])}")

Login OK - Token: YWM1MGQzZjAzMmM1YTlkYTA4MDQxNGQ0Yjk0NzA2YjU5NzQ1Y2ZjMWJVbzJaR3BhWVdGck9VZGphMHB4Vkd3MVF5OXFTbkZsYlRsSFVHeDBSMWxyU1c1U2Jsa3dQWHg4YlZweFVtMXdWRTE2YzI1UGRqWjVOblp5Y3oxOGZERjhmQT09...
Unidades do usuario: [{'Id': '110009826', 'Sigla': 'MUNICIPIO-PI/BOCAINA', 'Descricao': 'Prefeitura de Bocaina - MUNICIPIO-PI'}, {'Id': '110010294', 'Sigla': 'SAF-PI/GAB/DPEO', 'Descricao': 'Diretoria de Planejamento e Monitoramento - SAF-PI'}, {'Id': '110010629', 'Sigla': 'SEAD-PI/CIASPI/CSST-E-SOCIAL', 'Descricao': 'Coordenação de Saúde e Segurança do Trabalho  E-Social - SEAD-PI'}, {'Id': '110006253', 'Sigla': 'SEAD-PI/CIASPI/GPM/CM', 'Descricao': 'Gerência de Perícia Médica de Campo Maior - SEAD-PI'}, {'Id': '110006251', 'Sigla': 'SEAD-PI/CIASPI/GPM/FLO', 'Descricao': 'Gerência de Perícia Médica de Floriano - SEAD-PI'}, {'Id': '110006254', 'Sigla': 'SEAD-PI/CIASPI/GPM/PHB', 'Descricao': 'Gerência de Perícia Médica de Parnaíba - SEAD-PI'}, {'Id': '110006250', 'Sigla': 'SEAD-PI/CIASPI/GPM/

In [25]:
# Fetch all unidades from SEI
unidades_resp = requests.get(
    f"{SEI_BASE_URL}/unidades",
    params={"pagina": 1, "quantidade": 0},
    headers={"accept": "application/json", "token": token},
)
print(f"Status: {unidades_resp.status_code}")
unidades_data = unidades_resp.json()
qnt_unidades = unidades_data.get("Info").get("TotalItens", 0)
print(f"Total unidades: {qnt_unidades}")

Status: 200
Total unidades: 9393


In [26]:
unidades_resp = requests.get(
    f"{SEI_BASE_URL}/unidades",
    params={"pagina": 1, "quantidade": qnt_unidades},
    headers={"accept": "application/json", "token": token},
)
print(f"Status: {unidades_resp.status_code}")
unidades_data = unidades_resp.json()
print(f"Resposta: {unidades_data}")

unidades = unidades_data.get("Unidades", [])
print(f"Unidades encontradas: {len(unidades)}")
print(f"Primeiras 3 unidades: {unidades[:3] if isinstance(unidades, list) else unidades}")

Status: 200
Resposta: {'Info': {'Pagina': 1, 'TotalPaginas': 1, 'QuantidadeItens': 9393, 'TotalItens': 9393}, 'Unidades': [{'IdUnidade': '110010527', 'Sigla': '11BPM/3CIA/GPM/AnisiodeAbreu', 'Descricao': 'GPM de Anisio de Abreu PI', 'SinProtocolo': 'N', 'SinArquivamento': 'N', 'SinOuvidoria': 'N'}, {'IdUnidade': '110004367', 'Sigla': 'ADAPI-PI/ASDAPI-SIND', 'Descricao': 'Sindicato dos Servidores da Agência de Defesa Agropecuária do Estado do Piauí - ADAPI-PI', 'SinProtocolo': 'N', 'SinArquivamento': 'N', 'SinOuvidoria': 'N'}, {'IdUnidade': '110002059', 'Sigla': 'ADAPI-PI/DG', 'Descricao': 'Diretoria Geral - ADAPI-PI', 'SinProtocolo': 'S', 'SinArquivamento': 'S', 'SinOuvidoria': 'N'}, {'IdUnidade': '110010967', 'Sigla': 'ADAPI-PI/DG/ATENPGE', 'Descricao': 'Atendimento à Requisições e Diligências da Procuradoria de Geral Estado (PGE) - ADAPI-PI', 'SinProtocolo': 'N', 'SinArquivamento': 'N', 'SinOuvidoria': 'N'}, {'IdUnidade': '110002062', 'Sigla': 'ADAPI-PI/DG/DAF', 'Descricao': 'Diretor

In [27]:
# Create DataFrame and save to CSV
df_unidades = pd.DataFrame(unidades)
df_unidades.head(10)

,IdUnidade,Sigla,Descricao,SinProtocolo,SinArquivamento,SinOuvidoria
0,110010527,11BPM/3CIA/GPM/AnisiodeAbreu,GPM de Anisio de Abreu PI,N,N,N
1,110004367,ADAPI-PI/ASDAPI-SIND,Sindicato dos Servidores da Agência de Defesa ...,N,N,N
2,110002059,ADAPI-PI/DG,Diretoria Geral - ADAPI-PI,S,S,N
3,110010967,ADAPI-PI/DG/ATENPGE,Atendimento à Requisições e Diligências da Pro...,N,N,N
4,110002062,ADAPI-PI/DG/DAF,Diretoria Administrativa e Financeira - ADAPI-PI,N,N,N
5,110004369,ADAPI-PI/DG/DAF/ADM,Coordenação Administrativa - ADAPI-PI,N,N,N
6,110002067,ADAPI-PI/DG/DAF/ALMOX,Setor de Almoxarifado - ADAPI-PI,N,N,N
7,110002068,ADAPI-PI/DG/DAF/CF,Coordenação Financeira - ADAPI-PI,N,N,N
8,110006091,ADAPI-PI/DG/DAF/CF/ARQ,Arquivo - ADAPI-PI,N,N,N
9,110006092,ADAPI-PI/DG/DAF/CF/CONT,Contábil - ADAPI-PI,N,N,N


In [28]:
df_unidades.to_csv("unidades_sei.csv", index=False)
print(f"Saved {len(df_unidades)} unidades to unidades_sei.csv")

Saved 9393 unidades to unidades_sei.csv


In [ ]:
# Preparar sessão com retry e connection pooling
session = requests.Session()
retries = Retry(total=3, backoff_factor=0.3, status_forcelist=(500,502,504))
adapter = HTTPAdapter(max_retries=retries, pool_connections=100, pool_maxsize=100)
session.mount('https://', adapter)
session.mount('http://', adapter)

def fetch_users(id_unidade):
    try:
        # primeira chamada para obter total de usuários
        resp = session.get(f'{SEI_BASE_URL}/unidades/{id_unidade}/usuarios',
                           params={'pagina': 1, 'quantidade': 0},
                           headers={'accept': 'application/json', 'token': token},
                           timeout=10)
        resp.raise_for_status()
        data = resp.json()
        total = data.get('Info', {}).get('TotalItens', 0)
        if total and total > 0:
            resp = session.get(f'{SEI_BASE_URL}/unidades/{id_unidade}/usuarios',
                                params={'pagina': 1, 'quantidade': total},
                                headers={'accept': 'application/json', 'token': token},
                                timeout=30)
            resp.raise_for_status()
            data = resp.json()
        usuarios = data.get('Usuarios', []) if isinstance(data, dict) else []
        return {'IdUnidade': id_unidade, 'Usuarios': usuarios}
    except Exception as e:
        return {'IdUnidade': id_unidade, 'Usuarios': [], 'error': str(e)}

# Executar em paralelo com log a cada 500 unidades
ids = df_unidades['IdUnidade'].tolist()
total_ids = len(ids)
max_workers = min(50, max(5, (len(ids) // 2) or 5))
print('max_workers:', max_workers)
usuarios_por_caixa_resp = []
processed = 0
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(fetch_users, idu): idu for idu in ids}
    for future in as_completed(futures):
        usuarios_por_caixa_resp.append(future.result())
        processed += 1
        if processed % 500 == 0 or processed == total_ids:
            print(f'Processed {processed}/{total_ids} unidades')

: 

: 

In [ ]:
# Criar DataFrame plano (cada usuário vira uma linha) e salvar CSV
rows = []
for entry in usuarios_por_caixa_resp:
    id_un = entry.get('IdUnidade')
    for u in entry.get('Usuarios', []):
        row = {'IdUnidade': id_un}
        if isinstance(u, dict):
            row.update(u)
        rows.append(row)

df_usuarios = pd.DataFrame(rows)
print(f'Total usuários agregados: {len(df_usuarios)}')
df_usuarios.to_csv('usuarios_por_unidade.csv', index=False)
print('Saved usuarios_por_unidade.csv')